The goal of this notebook is to create an unconditional diffusion model from scratch

# Load deps

In [ ]:
# ! pip install -q torcheval

In [ ]:
! pip install -q torchview graphviz

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
import torch, os, math
import matplotlib as mpl
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from pathlib import Path
from scipy import integrate
from functools import partial
from torch import nn, optim, tensor
from torch.utils.data import DataLoader,default_collate
from torch.optim import lr_scheduler
from fastprogress import progress_bar
from datasets import load_dataset
from functools import wraps
from torchview import draw_graph

from src.utils import set_seed, noop
from src.datasets import DataLoaders, show_images, inplace, show_image
from src.learner import DeviceCB, ProgressCB, MetricsCB, Learner
from src.accel import MixedPrecision
from src.sgd import BatchSchedCB
from src.fid import ImageEval
from src.init import clean_mem

# Config

In [ ]:
# os.environ['CUDA_VISIBLE_DEVICES']='2' --> not needed if we have a single GPU
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)
mpl.rcParams['figure.dpi'] = 70
dataset_xl,dataset_yl = 'image','label'
dataset_name = "zalando-datasets/fashion_mnist"
set_seed(42)
bs = 512
n_steps = 1000
data_path = Path('artifacts/data')
data_path.mkdir(exist_ok=True, parents=True)
models_path = Path("artifacts/models")
models_path.mkdir(exist_ok=True, parents=True)
vis_path = Path("artifacts/visualization")
vis_path.mkdir(exist_ok=True, parents=True)

# Load dataset

In [ ]:
@inplace
def transformi(b):
    b[dataset_xl] = [
        F.pad(TF.to_tensor(o), (2,2,2,2))*2-1
        for o in b[dataset_xl]
    ]
    
dsd = load_dataset(dataset_name)
tds = dsd.with_transform(transformi)

In [ ]:
# just to calculate data std
dls = DataLoaders.from_dd(tds, bs)
dl = dls.train
xb,yb = b = next(iter(dl))
sig_data = xb.std()
print(f"xb.std: {sig_data}")

## Apply noisification

- we use the same noisifying method as karras notebook

In [ ]:
def scalings(sig):
    totvar = sig**2+sig_data**2
    c_skip = sig_data**2/totvar
    c_out = sig*sig_data/totvar.sqrt()
    c_in = 1/totvar.sqrt()
    if torch.isnan(c_skip).any() or torch.isnan(c_out).any() or torch.isnan(c_in).any():
        nan_str_skip = f"c_skip={sig[torch.isnan(c_skip)]}"
        nan_str_out = f"c_out={sig[torch.isnan(c_out)]}"
        nan_str_in = f"c_in={sig[torch.isnan(c_in)]}"
        print(f"found NaN values --> {nan_str_skip}, {nan_str_out}, {nan_str_in}")
    return c_skip,c_out,c_in

def noisify(x0):
    device = x0.device
    sig = (torch.randn([len(x0)])*1.2-1.2).exp().to(x0).reshape(-1,1,1,1)
    noise = torch.randn_like(x0, device=device)
    c_skip,c_out,c_in = scalings(sig)
    noised_input = x0 + noise*sig
    target = (x0-c_skip*noised_input)/c_out
    return (noised_input*c_in,sig.squeeze()),target

def collate_ddpm(b): return noisify(default_collate(b)[dataset_xl])

def dl_ddpm(ds): return DataLoader(
    ds, batch_size=bs,
    collate_fn=collate_ddpm,
    num_workers=os.cpu_count()
)


dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

# Basic Unet Diffuser

## Define model arch

In [ ]:
# this is the pre-activation convolution used for tinyimagenet in the previous notebooks
def unet_conv(ni, nf, ks=3, stride=1, act=nn.SiLU, norm=None, bias=True):
    layers = nn.Sequential()
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(
        nn.Conv2d(
            ni, nf, stride=stride, kernel_size=ks,
            padding=ks//2, bias=bias
        )
    )
    return layers

class UnetResBlock(nn.Module):
    def __init__(self, ni, nf=None, ks=3, act=nn.SiLU, norm=nn.BatchNorm2d):
        # no option to add down-sampling here.
        # down-sampling is handled by down_block, defined below
        super().__init__()
        if nf is None: nf = ni
        self.convs = nn.Sequential(
            unet_conv(ni, nf, ks, act=act, norm=norm),
            unet_conv(nf, nf, ks, act=act, norm=norm)
        )
        self.idconv = noop if ni==nf else nn.Conv2d(ni, nf, 1)

    def forward(self, x): return self.convs(x) + self.idconv(x)

### Mixin design pattern to create down path blocks

In [ ]:
class A:
    def __call__(self):
        super().__call__()
        print('a')

class B:
    def __call__(self): print('b')

class C(A,B): pass
C()()

In [ ]:
class SaveModule:
    def forward(self, x, *args, **kwargs):
        self.saved = super().forward(x, *args, **kwargs)
        return self.saved

class SavedResBlock(SaveModule, UnetResBlock): pass
class SavedConv(SaveModule, nn.Conv2d): pass

In [ ]:
def down_block(ni, nf, add_down=True, num_layers=1):
    res = nn.Sequential(
        *[
            SavedResBlock(ni=ni if i==0 else nf, nf=nf)
            for i in range(num_layers)
        ]
    )
    if add_down: res.append(SavedConv(nf, nf, 3, stride=2, padding=1))
    return res

In [ ]:
def upsample(nf):
    return nn.Sequential(
        nn.Upsample(scale_factor=2.),
        nn.Conv2d(nf, nf, 3, padding=1)
    )

class UpBlock(nn.Module):
    def __init__(self, ni, prev_nf, nf, add_up=True, num_layers=2):
        super().__init__()
        self.resnets = nn.ModuleList(
            [UnetResBlock((prev_nf if i==0 else nf)+(ni if (i==num_layers-1) else nf), nf)
            for i in range(num_layers)])
        self.up = upsample(nf) if add_up else nn.Identity()

    def forward(self, x, ups):
        # ups is a list of saved activations during the down path
        # here we're using concatenation instead of summation
        for resnet in self.resnets: x = resnet(torch.cat([x, ups.pop()], dim=1))
        return self.up(x)

In [ ]:
class UNet2DModel(nn.Module):
    def __init__( self, in_channels=3, out_channels=3, nfs=(224,448,672,896), num_layers=1):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, nfs[0], kernel_size=3, padding=1)
        nf = nfs[0]
        self.downs = nn.Sequential()
        for i in range(len(nfs)):
            ni = nf
            nf = nfs[i]
            self.downs.append(down_block(ni, nf, add_down=i!=len(nfs)-1, num_layers=num_layers))
        # we didn't have this before. introduced here
        self.mid_block = UnetResBlock(nfs[-1])

        rev_nfs = list(reversed(nfs))
        nf = rev_nfs[0]
        self.ups = nn.ModuleList()
        for i in range(len(nfs)):
            prev_nf = nf
            nf = rev_nfs[i]
            ni = rev_nfs[min(i+1, len(nfs)-1)]
            self.ups.append(UpBlock(ni, prev_nf, nf, add_up=i!=len(nfs)-1, num_layers=num_layers+1))
            # the extra layer here is because in each down_block,
            # we have an extra conv at the end that save its results as well.
        self.conv_out = unet_conv(nfs[0], out_channels, act=nn.SiLU, norm=nn.BatchNorm2d)

    def forward(self, inp):
        x = self.conv_in(inp[0])
        saved = [x]
        x = self.downs(x)
        saved += [p.saved for o in self.downs for p in o]
        x = self.mid_block(x)
        for block in self.ups: x = block(x, saved)
        return self.conv_out(x)

## Inspect model arch + DAG

In [ ]:
# from torchview import draw_graph
# model = UNet2DModel(in_channels=1, out_channels=1, nfs=(32,64), num_layers=2)

# xb, yb = next(iter(dls.train))
# model_graph = draw_graph(
#     model,
#     input_data=[(xb[0][:1], xb[1][:1])],
#     expand_nested=True,
#     depth=3,
#     hide_module_functions=False
# )
# model_graph.visual_graph.render(vis_path / "model_graph", format="png")

## Train process

In [ ]:
model = UNet2DModel(in_channels=1, out_channels=1, nfs=(32,64,128,256), num_layers=2)
lr = 3e-3
epochs = 2
opt_func = partial(optim.Adam, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [
    DeviceCB(), MixedPrecision(), ProgressCB(plot=True),
    MetricsCB(), BatchSchedCB(sched)
]
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [ ]:
# clean_mem()
# learn.fit(epochs)

# Timesteps Model

## Timesteps

In [ ]:
emb_dim = 16
tsteps = torch.linspace(-10,10,100)
max_period = 10000

fig, axs = plt.subplots(1,3, figsize=(24,6))


exponent = -math.log(max_period) * torch.linspace(0, 1, emb_dim//2, device=tsteps.device)
axs[0].plot(exponent);
axs[0].set_title(f"exponents | shape: {exponent.shape}")


emb = tsteps[:,None].float() * exponent.exp()[None,:]
for idx in [0,10,20,50,99]:
    axs[1].plot(emb[idx], label = f"emb_tstep_({torch.round(tsteps[idx])})")
axs[1].set_title(f"embeddings | shape: {emb.shape}")
axs[1].legend();


emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
for idx in [0,1,2,3,8,9,10,11]:
    axs[2].plot(emb[:,idx], label = f"emb_col_{idx}")
axs[2].set_title(f"embeddings | shape: {emb.shape}")
axs[2].legend();

In [ ]:
show_image(emb.T, figsize=(7,7));

In [ ]:
def timestep_embedding(tsteps, emb_dim, max_period= 10000):
    exponent = -math.log(max_period) * torch.linspace(0, 1, emb_dim//2, device=tsteps.device)
    emb = tsteps[:,None].float() * exponent.exp()[None,:]
    emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
    return F.pad(emb, (0,1,0,0)) if emb_dim%2==1 else emb

In [ ]:
show_image(timestep_embedding(tsteps, 32, max_period=1000).T, figsize=(7,7));

In [ ]:
show_image(timestep_embedding(tsteps, 32, max_period=10).T, figsize=(7,7));

## Define model arch

In [ ]:
def lin(ni, nf, act=nn.SiLU, norm=None, bias=True):
    layers = nn.Sequential()
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(nn.Linear(ni, nf, bias=bias))
    return layers

In [ ]:
class EmbResBlock(nn.Module):
    def __init__(self, n_emb, ni, nf=None, ks=3, act=nn.SiLU, norm=nn.BatchNorm2d):
        super().__init__()
        if nf is None: nf = ni
        self.emb_proj = nn.Linear(n_emb, nf*2)
        # 2x the number of output so that the torch.chunk below works
        # half of it used as shift and the other half as scale
        self.conv1 = unet_conv(ni, nf, ks, act=act, norm=norm) #, bias=not norm)
        self.conv2 = unet_conv(nf, nf, ks, act=act, norm=norm)
        self.idconv = noop if ni==nf else nn.Conv2d(ni, nf, 1)

    def forward(self, x, t):
        inp = x
        x = self.conv1(x)
        emb = self.emb_proj(F.silu(t))[:, :, None, None] # add asex to convert (N,C) --> (N,C,H,W)
        # TODO: why do we generate negative time values to be damped by this activation?
        # it's like we produce negative values and we also discard them. why??
        scale,shift = torch.chunk(emb, 2, dim=1)
        x = x*(1+scale) + shift
        x = self.conv2(x)
        return x + self.idconv(inp)

### Using decorator/monkey patch instead of Mixin pattern

In [ ]:
def saved(m, blk):
    m_ = m.forward

    @wraps(m.forward)
    def _f(*args, **kwargs):
        res = m_(*args, **kwargs)
        blk.saved.append(res)
        return res

    m.forward = _f
    return m

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, n_emb, ni, nf, add_down=True, num_layers=1):
        super().__init__()
        self.resnets = nn.ModuleList(
            [
                saved(EmbResBlock(n_emb, ni if i==0 else nf, nf), self) # we pass the self/block to saved
                for i in range(num_layers)
            ]
        )
        self.down = saved(nn.Conv2d(nf, nf, 3, stride=2, padding=1), self) if add_down else nn.Identity()

    def forward(self, x, t):
        # TODO: we can't use default sequential anymore because we feed t as well to the input
        # we can create a custom sequential and reuse it.
        self.saved = []
        for resnet in self.resnets: x = resnet(x, t)
        x = self.down(x)
        return x

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, n_emb, ni, prev_nf, nf, add_up=True, num_layers=2):
        super().__init__()
        self.resnets = nn.ModuleList(
            [EmbResBlock(n_emb, (prev_nf if i==0 else nf)+(ni if (i==num_layers-1) else nf), nf)
            for i in range(num_layers)]
        )
        self.up = upsample(nf) if add_up else nn.Identity()

    def forward(self, x, t, ups):
        for resnet in self.resnets: x = resnet(torch.cat([x, ups.pop()], dim=1), t)
        return self.up(x)

In [ ]:
class EmbUNetModel(nn.Module):
    def __init__( self, in_channels=3, out_channels=3, nfs=(224,448,672,896), num_layers=1):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, nfs[0], kernel_size=3, padding=1)
        self.n_temb = nf = nfs[0]
        n_emb = nf*4
        # TODO: remove act func from 1st MLP layer
        self.emb_mlp = nn.Sequential(lin(self.n_temb, n_emb, norm=nn.BatchNorm1d),
                                     lin(n_emb, n_emb))
        self.downs = nn.ModuleList()
        for i in range(len(nfs)):
            ni = nf
            nf = nfs[i]
            self.downs.append(DownBlock(n_emb, ni, nf, add_down=i!=len(nfs)-1, num_layers=num_layers))
        self.mid_block = EmbResBlock(n_emb, nfs[-1])

        rev_nfs = list(reversed(nfs))
        nf = rev_nfs[0]
        self.ups = nn.ModuleList()
        for i in range(len(nfs)):
            prev_nf = nf
            nf = rev_nfs[i]
            ni = rev_nfs[min(i+1, len(nfs)-1)]
            self.ups.append(UpBlock(n_emb, ni, prev_nf, nf, add_up=i!=len(nfs)-1, num_layers=num_layers+1))
        self.conv_out = unet_conv(nfs[0], out_channels, act=nn.SiLU, norm=nn.BatchNorm2d, bias=False)

    def forward(self, inp):
        x,t = inp # we use t as well as input now
        temb = timestep_embedding(t, self.n_temb)
        emb = self.emb_mlp(temb)
        x = self.conv_in(x)
        saved = [x]
        for block in self.downs: x = block(x, emb)
        #TODO: do we need to pass emb every time??
        # for example, NLP models inject the position embeddings only once at the beginning
        saved += [p for o in self.downs for p in o.saved]
        x = self.mid_block(x, emb)
        for block in self.ups: x = block(x, emb, saved)
        return self.conv_out(x)

## Inspect model arch + DAG

In [ ]:
model = EmbUNetModel(in_channels=1, out_channels=1, nfs=(32,64), num_layers=1)

xb, yb = next(iter(dls.train))
model_graph = draw_graph(
    model,
    input_data=[(xb[0][:1], xb[1][:1])],
    expand_nested=False,
    depth=2,
    hide_module_functions=False
)
model_graph.visual_graph.render(vis_path / "model_graph", format="png")

## Train Process

In [ ]:
model = EmbUNetModel(in_channels=1, out_channels=1, nfs=(32,64,128,256), num_layers=2)
lr = 1e-2
epochs = 25
opt_func = partial(optim.Adam, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), MixedPrecision()]
model = EmbUNetModel(in_channels=1, out_channels=1, nfs=(32,64,128,256), num_layers=2)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [ ]:
# clean_mem()
# learn.fit(epochs)

In [ ]:
# torch.save(learn.model.state_dict(), models_path / 'unconditional-unet-diffuser.pkl')

In [ ]:
learn.model.load_state_dict(torch.load(models_path / 'unconditional-unet-diffuser.pkl'))